In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json, os
from pathlib import Path
from ecosentry.arch6_payload import AlertPayloadCodec, DeviceMetadata
from ecosentry.contracts import Arch5Result
from ecosentry.config import load_canonical_config

config = load_canonical_config()
encryption_key = os.urandom(32)
hmac_key = os.urandom(32)
codec = AlertPayloadCodec(config, encryption_key, hmac_key)

arch5_res = Arch5Result(timestamp_ms=1700000000000, class_id=1, confidence=0.95)
meta = DeviceMetadata(device_id="node-xyz", location_hash="geo-hash", firmware_version="2.0.1")

encoded = codec.encode(arch5_res, meta)
decoded = codec.decode(encoded.encrypted_payload_bytes)

res_dict = {
    "size_report": {"raw": encoded.size_report.raw_json_bytes, "compressed": encoded.size_report.compressed_bytes, "encrypted": encoded.size_report.encrypted_bytes, "enveloped": encoded.size_report.enveloped_bytes},
    "envelope_metadata": codec.metadata_to_dict(encoded.envelope_metadata),
    "decoded": decoded
}

Path("results").mkdir(exist_ok=True)
with open("results/arch6_report.json", "w") as f:
    json.dump(res_dict, f, indent=2)
print("Decoded Payload:", decoded)
print("Size Report:", res_dict["size_report"])

sizes = [res_dict["size_report"]["raw"], res_dict["size_report"]["compressed"], res_dict["size_report"]["encrypted"], res_dict["size_report"]["enveloped"]]
labels = ["Raw JSON", "Compressed", "Encrypted", "Enveloped"]

plt.figure(figsize=(8, 5))
sns.barplot(x=labels, y=sizes, palette="viridis")
plt.title("Payload Size Progression (Bytes)")
plt.ylabel("Bytes")
plt.show()